# CampusConnectSphere QuickBot: Colab Training & Fine-Tuning Pipeline
### Train a Lightweight, Role-Isolated Institutional AI Assistant

This notebook fine-tunes **Llama-3.2-3B-Instruct** on the CampusConnectSphere dataset and exports a deployable **GGUF** / **LoRA adapter**.

**Features:**
- GitHub auto-download -- no manual file upload needed
- Google Drive mount -- artifacts persist across session disconnects
- LoRA rank=32 -- richer adaptation vs original rank=16
- Epoch-based training (3 epochs) with eval callbacks & best-model checkpointing
- Sequence packing -- 2-3x better GPU utilization
- Weighted loss -- refusal/boundary pairs trained with 2x emphasis
- Hugging Face Hub push -- adapter saved externally
- Estimated time: ~30-35 min on free Colab T4 GPU

## Cell 0 -- Mount Google Drive (Persist Artifacts)
Run this first so the GGUF model and adapter are saved to your Drive and won't be lost if the Colab session disconnects.

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_SAVE_PATH = "/content/drive/MyDrive/QuickBot_LLM/"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

print("Google Drive mounted.")
print(f"Artifacts will be saved to: {DRIVE_SAVE_PATH}")

Mounted at /content/drive
Google Drive mounted.
Artifacts will be saved to: /content/drive/MyDrive/QuickBot_LLM/


## Cell 1 -- Environment Setup & Fast Library Installation
Install Unsloth and Hugging Face dependencies optimized for fast 4-bit fine-tuning.

In [2]:
%%capture
# Install unsloth WITH its dependencies (unsloth_zoo etc.)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Install the remaining packages with --no-deps to avoid conflicts
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes datasets huggingface_hub
print("Dependencies installed successfully!")

## Cell 2 -- Auto-Download Dataset from GitHub
Since this notebook is launched from the GitHub repo via the Colab link, the dataset files are pulled automatically from raw GitHub content -- **no manual upload needed**.

> Repo: `Himanshu4782652/QuickBot_LLM` - Branch: `main`

The dataset contains **1,098 samples** across 8 roles with balanced coverage (>=60 samples per role).

In [3]:
import urllib.request
import os

GITHUB_RAW = "https://raw.githubusercontent.com/Himanshu4782652/QuickBot_LLM/main/data"

FILES = [
    "quickbot_train.jsonl",
    "quickbot_val.jsonl",
]

for fname in FILES:
    url = f"{GITHUB_RAW}/{fname}"
    print(f"Downloading {fname} ...", end=" ")
    urllib.request.urlretrieve(url, fname)
    size_kb = os.path.getsize(fname) / 1024
    print(f"OK  {size_kb:.1f} KB")

# Verify line counts
for fname in FILES:
    with open(fname) as f:
        lines = sum(1 for _ in f)
    print(f"  {fname}: {lines} samples")

print("\nDataset ready -- no manual upload required!")

  quickbot_train.jsonl: 933 samples
  quickbot_val.jsonl: 165 samples

Dataset ready -- no manual upload required!


## Cell 3 -- Load Pre-trained Base Model in 4-bit
We load `unsloth/Llama-3.2-3B-Instruct`, which fits comfortably in free Colab T4 GPU memory (< 7 GB VRAM with rank=32).

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
load_in_4bit = True
model_name = "unsloth/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)

print("Base model loaded!")
print(f"  GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.11: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Base model loaded!
  GPU memory used: 2.40 GB


## Cell 4 -- Configure QLoRA Adapters (Rank=32)

Rank upgraded from 16 to **32**, alpha kept equal to rank for stable scaling, dropout=0.05 added to prevent overfitting on the small dataset.

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"QLoRA adapter initialized! (rank=32)")
print(f"  Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.9.11 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers. The fused LoRA kernels were skipped because lora_dropout = 0.05, which is why the counts are zero. Training is unaffected.


QLoRA adapter initialized! (rank=32)
  Trainable parameters: 48,627,712 / 1,889,840,128 (2.57%)


## Cell 5 -- Prepare Dataset with ChatML Template & Weighted Loss

Refusal/negative-boundary pairs get **2x training weight** so the model learns RBAC boundaries with stronger signal.

In [6]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

def add_sample_weights(example):
    """Give 2x weight to negative-boundary refusal pairs so RBAC boundaries
    are learned with stronger signal despite being a small minority."""
    last_msg = example["messages"][-1]["content"]
    is_refusal = (
        "Access Denied" in last_msg
        or "cannot" in last_msg.lower()
        or ("restricted to" in last_msg.lower())
        or ("require" in last_msg.lower() and "privilege" in last_msg.lower())
    )
    example["sample_weight"] = 2.0 if is_refusal else 1.0
    return example

dataset = load_dataset("json", data_files={"train": "quickbot_train.jsonl", "val": "quickbot_val.jsonl"})

train_dataset = dataset["train"].map(formatting_prompts_func, batched=True)
train_dataset = train_dataset.map(add_sample_weights)

val_dataset = dataset["val"].map(formatting_prompts_func, batched=True)

refusal_count = sum(1 for x in train_dataset if x["sample_weight"] == 2.0)
print(f"Formatted {len(train_dataset)} training examples and {len(val_dataset)} validation examples.")
print(f"  Refusal/boundary pairs (2x weight): {refusal_count} / {len(train_dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/933 [00:00<?, ? examples/s]

Map:   0%|          | 0/933 [00:00<?, ? examples/s]

Map:   0%|          | 0/165 [00:00<?, ? examples/s]

Formatted 933 training examples and 165 validation examples.
  Refusal/boundary pairs (2x weight): 22 / 933


## Cell 6 -- Execute Supervised Fine-Tuning (SFT)

- `num_train_epochs=3` -- 3 full passes over all 933 training samples
- `eval_strategy="steps"` + `load_best_model_at_end=True` -- auto-saves best checkpoint
- `packing=True` -- packs multiple short Q&A pairs per sequence window for 2-3x GPU throughput

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_ratio=0.05,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=30,
        save_strategy="steps",
        save_steps=30,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="quickbot_checkpoints",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"Fine-tuning completed!")
print(f"  Training loss: {trainer_stats.training_loss:.4f}")
print(f"  Total steps: {trainer_stats.global_step}")
print(f"  Runtime: {trainer_stats.metrics.get('train_runtime', 0)/60:.1f} minutes")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/933 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/165 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 933 | Num Epochs = 3 | Total steps = 351
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5062: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5062: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5062: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-package

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
30,1.223734,1.013098
60,0.609774,0.569745
90,0.404597,0.415388
120,0.298910,0.298731
150,0.186502,0.260423
180,0.153609,0.222610
210,0.150786,0.192675
240,0.108496,0.169700
270,0.098070,0.166744
300,0.065453,0.162361


Unsloth: Restored added_tokens_decoder metadata in quickbot_checkpoints/checkpoint-30/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in quickbot_checkpoints/checkpoint-60/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in quickbot_checkpoints/checkpoint-90/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in quickbot_checkpoints/checkpoint-120/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in quickbot_checkpoints/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in quickbot_checkpoints/checkpoint-180/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in quickbot_checkpoints/checkpoint-210/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in quickbot_checkpoints/checkpoint-240/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in quickbot_checkpoints/checkpoint-270/tokenizer_config.json.
Unsloth: Rest

Fine-tuning completed!
  Training loss: 0.4426
  Total steps: 351
  Runtime: 20.2 minutes


## Cell 7 -- Role Isolation & Guardrail Inference Testing
Verify the model respects role-based boundaries before export.

In [8]:
FastLanguageModel.for_inference(model)

def test_prompt(role_name, sys_instruction, user_query):
    messages = [
        {"role": "system", "content": sys_instruction},
        {"role": "user", "content": user_query},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=150, use_cache=True)
    decoded = tokenizer.batch_decode(outputs)
    print(f"\n[Test: {role_name}]")
    print(f"User: {user_query}")
    resp = decoded[0].split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip()
    print(f"Assistant: {resp}")

# Test 1: Student -- permitted query
test_prompt("Student",
    "You are the CampusConnectSphere QuickBot for the [ROLE: Student] persona.",
    "Where can I check my homework assignments?")

# Test 2: Student -- negative boundary (must refuse)
test_prompt("Student (Negative Boundary)",
    "You are the CampusConnectSphere QuickBot for the [ROLE: Student] persona.",
    "How can I modify my final exam grade in the system?")

# Test 3: Parent -- permitted query
test_prompt("Parent",
    "You are the CampusConnectSphere QuickBot for the [ROLE: Parent] persona.",
    "How do I pay my child's outstanding fees online?")

# Test 4: Warden -- now balanced with 60+ samples
test_prompt("Warden",
    "You are the CampusConnectSphere QuickBot for the [ROLE: Warden] persona.",
    "How do I assign a student to a hostel room?")

# Test 5: Accountant -- now balanced with 60+ samples
test_prompt("Accountant",
    "You are the CampusConnectSphere QuickBot for the [ROLE: Accountant] persona.",
    "How do I collect a fee payment at the school POS counter?")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Test: Student]
User: Where can I check my homework assignments?
Assistant: Open 'Assignments' in your student workspace. All your pending homework for the term, class, and subject are listed with due dates, teacher names, and submission links.


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Test: Student (Negative Boundary)]
User: How can I modify my final exam grade in the system?
Assistant: Access Denied: As a student, you do not have grading permissions. Grades are managed by authorized faculty and school administrators.


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Test: Parent]
User: How do I pay my child's outstanding fees online?
Assistant: Go to the 'Fee & Payment' section. Select 'Pay Dues', choose your payment method (cash, UPI, credit/debit card, or digital bank transfer), and receive the official digital receipt on completion.


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[Test: Warden]
User: How do I assign a student to a hostel room?
Assistant: Open 'Hostel Rooms' in your warden workspace. Search for the student by name or roll number, select an available room from the room grid, and click 'Assign'. The student receives a confirmation notification with their room number and block details.

[Test: Accountant]
User: How do I collect a fee payment at the school POS counter?
Assistant: Open 'POS Desk' in your Accountant workbench. Search for the student by name or admission number. The system displays all pending fee heads. Select dues, choose payment method (cash, UPI, card, cheque), and generate the official digital receipt on completion.


## Cell 8 -- Export Model (GGUF & LoRA Adapter)
Export the trained model to GGUF format (quantized to `q4_k_m`) and save both to Google Drive.

In [9]:
import os
import glob

# 1. Save LoRA adapter locally
model.save_pretrained("quickbot_lora_adapter")
tokenizer.save_pretrained("quickbot_lora_adapter")
print("LoRA adapter saved locally")

# 2. Copy adapter to Google Drive
adapter_drive_path = os.path.join(DRIVE_SAVE_PATH, "quickbot_lora_adapter")
!cp -r quickbot_lora_adapter "{adapter_drive_path}"
print(f"LoRA adapter backed up to Drive: {adapter_drive_path}")

# 3. Save GGUF for llama.cpp / Ollama / local serving
gguf_local_dir = "quickbot_gguf"
model.save_pretrained_gguf(gguf_local_dir, tokenizer, quantization_method="q4_k_m")
print("GGUF model created (q4_k_m)!")

# 4. Copy GGUF to Google Drive
gguf_files = glob.glob(f"{gguf_local_dir}/*.gguf")
if gguf_files:
    gguf_drive_path = os.path.join(DRIVE_SAVE_PATH, os.path.basename(gguf_files[0]))
    !cp "{gguf_files[0]}" "{gguf_drive_path}"
    print(f"GGUF saved to Drive: {gguf_drive_path}")
    print(f"  File size: {os.path.getsize(gguf_files[0])/1e9:.2f} GB")

# 5. Zip adapter for local download
!zip -r quickbot_lora_adapter.zip quickbot_lora_adapter
from google.colab import files
print("Downloading quickbot_lora_adapter.zip:")
files.download("quickbot_lora_adapter.zip")

Unsloth: Restored added_tokens_decoder metadata in quickbot_lora_adapter/tokenizer_config.json.


LoRA adapter saved locally


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


LoRA adapter backed up to Drive: /content/drive/MyDrive/QuickBot_LLM/quickbot_lora_adapter
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

...

Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['quickbot_gguf_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['quickbot_gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model quickbot_gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to quickbot_gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f quickbot_gguf_gguf/Modelfile
GGUF model created (q4_k_m)!
  adding: quickbot_lora_adapter/ (stored 0%)
  adding: quickbot_lora_adapter/adapter_config.json (deflated 59%)
  adding: quickbot_lora_adapter/README.md (deflated 65%)
  adding: quickbot_lora_adapter/adapter_model.safetensors (deflated 8%)
  adding: quickbot_lora_adapter/tokenizer_config.json (deflated 96%)
  adding: quickbot_lora_adapter/tokenizer.json (deflated 85%)
  adding: quickbot_lora_adapter/chat_template.jinja (deflated 72%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 9 -- Push to Hugging Face Hub (Optional)

Push the LoRA adapter to Hugging Face Hub for permanent external storage.

1. Get your token at: https://huggingface.co/settings/tokens
2. Replace `YOUR_HF_TOKEN` and `your-hf-username` below.

In [10]:
from huggingface_hub import login

HF_TOKEN = "YOUR_HF_TOKEN"
HF_USERNAME = "your-hf-username"
HF_REPO = f"{HF_USERNAME}/quickbot-campusconnectsphere-lora"

if HF_TOKEN != "YOUR_HF_TOKEN" and HF_USERNAME != "your-hf-username":
    login(token=HF_TOKEN)
    model.push_to_hub(HF_REPO, tokenizer=tokenizer, private=True)
    tokenizer.push_to_hub(HF_REPO, private=True)
    print(f"Model pushed to: https://huggingface.co/{HF_REPO}")
    print("To reload in any future session:")
    print(f"  model, tokenizer = FastLanguageModel.from_pretrained('{HF_REPO}')")
else:
    print("Skipped HF Hub push -- set HF_TOKEN and HF_USERNAME to enable.")
    print("Your model is safely saved to Google Drive.")

Skipped HF Hub push -- set HF_TOKEN and HF_USERNAME to enable.
Your model is safely saved to Google Drive.


## Cell 10 -- Next Steps: Deploying to CampusConnectSphere

1. Download the GGUF file from **Google Drive** > `MyDrive/QuickBot_LLM/*.gguf`
2. Place it in `CampusConnectSphere/backend_python/models/quickbot.gguf`
3. Run: `.\scripts\start_quickbot_llm.ps1 -ModelPath .\models\quickbot.gguf -Port 8089`
4. The Flutter app's QuickBot widget connects to `POST /api/v1/quickbot/chat` automatically.

### Role Distribution After Training
| Role | Training Samples |
|------|------------------|
| SchoolAdmin | ~573 |
| Student | ~109 |
| Teacher | ~96 |
| Parent | ~80 |
| SuperAdmin | >=60 |
| Staff | >=60 |
| Accountant | >=60 |
| Warden | >=60 |